# 37 ScienceWorld physical-action pilot (CPU only)
Run these cells in order in a **fresh CPU Colab runtime**. No Hugging Face token or GPU is required. This is a bounded diagnostic run, not a labelled dataset. It attempts up to 20 candidates; fewer may be discovered. Every completed candidate is saved to Drive.

Full simulator-state coverage and the partial-restoration definition remain unresolved. The comparison here is explicitly limited to exported objects and visible interface state. No final reversibility labels are produced.


In [ ]:
import sys, subprocess, os
os.environ['JAVA_TOOL_OPTIONS'] = '-Xlog:os+container=off'
subprocess.run(['java', '-version'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
 'git+https://github.com/allenai/ScienceWorld.git@e8216d6044e8e39be9fcb185e3b2dfb602584b52'], check=True)
print('Pinned ScienceWorld installed. No model packages needed.')


In [ ]:
from google.colab import drive
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ADNN_Irreversibility')
assert PROJECT.is_dir(), 'Project folder not found on Drive.'
print('Project:', PROJECT)


In [ ]:
from pathlib import Path
Path("/content/37_scienceworld_physical_pilot_colab.py").write_text('"""CPU diagnostic only. No final labels; no approximate or time-matched equality."""\nimport argparse\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport inspect\nimport json\nimport os\nimport platform\nimport time\nimport traceback\nfrom collections import deque, Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nCOMMIT = \'e8216d6044e8e39be9fcb185e3b2dfb602584b52\'\nJAR_SHA = \'e77b0fee7d68abe3ca5b12e57d86e2bfe7603f20c5200da0b0f085939faeb465\'\n\ndef save(path, value):\n    data = json.dumps(value, indent=2, allow_nan=False)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(data)\n    tmp.replace(path)\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\ndef physical(action):\n    return action.lower().startswith((\'activate \', \'deactivate \', \'mix \', \'pour \', \'move \', \'use \'))\n\ndef projected(snapshot):\n    # Diagnostic exported world only. Clock/history/goal progress remain separate.\n    return {k: snapshot[k] for k in (\'tree\', \'look\', \'inventory\', \'valid_actions\')}\n\ndef search(replay, prefix, candidate, reference, max_nodes, seconds):\n    """BFS over the enumerated valid interface; budgets never imply irreversibility."""\n    queue = deque([[]]); nodes = []; start = time.monotonic(); witness = None\n    reason = \'enumerated_space_exhausted\'\n    while queue:\n        if len(nodes) >= max_nodes or time.monotonic() - start >= seconds:\n            reason = \'budget_exhausted\'; break\n        undo = queue.popleft()\n        end, trace = replay(prefix + [candidate] + undo)\n        matches = projected(end) == projected(reference)\n        nodes.append({\'undo\': undo, \'snapshot_sha256\': digest(end),\n                      \'exported_world_matches\': matches, \'done\': end[\'done\'], \'trace\': trace})\n        if matches:\n            repeat, repeat_trace = replay(prefix + [candidate] + undo)\n            if repeat != end or repeat_trace != trace:\n                reason = \'witness_replay_unstable\'; break\n            witness = undo; reason = \'exported_world_witness_only\'; break\n        if len(undo) < 3 and not end[\'done\']:\n            queue.extend(undo + [a] for a in end[\'valid_actions\'])\n    return {\'status\': reason, \'nodes_tested\': len(nodes), \'queued_remaining\': len(queue),\n            \'witness\': witness, \'elapsed_seconds\': round(time.monotonic()-start, 3),\n            \'final_label\': None, \'nodes\': nodes,\n            \'coverage_caveat\': \'Enumerated valid interface only; full simulator state coverage unverified.\'}\n\ndef run(args):\n    option = \'-Xlog:os+container=off\'\n    if option not in os.environ.get(\'JAVA_TOOL_OPTIONS\', \'\').split():\n        os.environ[\'JAVA_TOOL_OPTIONS\'] = (os.environ.get(\'JAVA_TOOL_OPTIONS\', \'\')+\' \'+option).strip()\n    from scienceworld import ScienceWorldEnv\n    from scienceworld.constants import JAR_PATH\n    dist = importlib.metadata.distribution(\'scienceworld\')\n    source = json.loads(dist.read_text(\'direct_url.json\') or \'{}\')\n    assert source.get(\'vcs_info\', {}).get(\'commit_id\') == COMMIT, \'Install the pinned commit using the notebook setup cell.\'\n    assert hashlib.sha256(Path(JAR_PATH).read_bytes()).hexdigest() == JAR_SHA, \'JAR differs from prior pilot. Stop and share this error.\'\n    root = Path(args.project)\n    assert root.is_dir(), \'Mount Drive first; project folder is missing.\'\n    out = root/\'results\'/(\'scienceworld_physical_pilot_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'))\n    out.mkdir(parents=True)\n    save(out/\'config.json\', {\'commit\': COMMIT, \'jar_sha256\': JAR_SHA, \'version\': dist.version,\n        \'python\': platform.python_version(), \'arguments\': vars(args), \'simplifications\': \'\',\n        \'scope\': \'Physical-action diagnostics, not dataset labels\', \'comparison\': \'Exact exported tree plus look/inventory/valid actions\',\n        \'excluded_from_restoration_comparison\': [\'moves\', \'goal_progress\', \'score\', \'history\', \'hidden simulator internals\'],\n        \'full_state_coverage_verified\': False, \'partial_recovery_definition\': None})\n    (out/\'runner.py\').write_bytes(Path(__file__).read_bytes())\n    (out/\'installed_wrapper.py\').write_text(inspect.getsource(ScienceWorldEnv))\n    env = None; results = []; errors = []; discoveries = []\n    started = time.monotonic()\n    def summary():\n        return {\'diagnostic_run_complete\': True, \'candidate_checks\': len(results),\n            \'stable_candidates\': sum(r.get(\'replay_stable\', False) for r in results),\n            \'search_status_counts\': dict(Counter(r.get(\'search\', {}).get(\'status\', \'not_run\') for r in results)),\n            \'tasks_attempted\': [d[\'task\'] for d in discoveries], \'errors\': errors,\n            \'elapsed_minutes\': round((time.monotonic()-started)/60, 2), \'final_labels_created\': 0,\n            \'full_state_coverage_verified\': False, \'scienceworld_requirement_met\': False, \'saved_to\': str(out)}\n    try:\n        env = ScienceWorldEnv(envStepLimit=250)\n        tasks = env.get_task_names(); save(out/\'available_tasks.json\', tasks)\n        chosen = []\n        for keyword in (\'boil\', \'freeze\', \'mix\', \'melt\'):\n            match = next((t for t in tasks if keyword in t.lower() and t not in chosen), None)\n            if match: chosen.append(match)\n        if not chosen: raise RuntimeError(\'No physical task names matched; inspect available_tasks.json.\')\n        for task in chosen:\n            discovery = {\'task\': task, \'variation\': 0, \'candidates\': []}; discoveries.append(discovery)\n            folder = out/(\'task_\'+str(len(discoveries))); folder.mkdir()\n            try:\n                print(\'Discovering physical actions:\', task, flush=True)\n                env.load(task, 0, \'\', generateGoldPath=True)\n                gold = env.get_gold_action_sequence()\n                save(folder/\'gold_path.json\', gold)\n                if not gold or any(a.startswith(\'ERROR\') for a in gold):\n                    raise RuntimeError(\'Gold trajectory unavailable; no candidate actions invented.\')\n                def snap(done=False):\n                    return {\'tree\': env.getObjectTree(), \'look\': env.look(), \'inventory\': env.inventory(),\n                        \'valid_actions\': sorted(set(env.get_valid_action_object_combinations())),\n                        \'moves\': int(env.get_num_moves()), \'score\': float(env.lastStepScore),\n                        \'goal_progress\': env.get_goal_progress(), \'done\': bool(done)}\n                def replay(actions):\n                    env.load(task, 0, \'\', generateGoldPath=False)\n                    env.reset(); trace = []; done = False\n                    for action in actions:\n                        if done: raise RuntimeError(\'Replay would step beyond terminal state.\')\n                        obs, reward, done, info = env.step(action)\n                        trace.append({\'action\': action, \'observation\': obs, \'reward\': reward,\n                                      \'done\': bool(done), \'moves\': int(env.get_num_moves())})\n                    return snap(done), trace\n                # Follow supplied trajectory to reach physical situations; never infer success from text alone.\n                env.load(task, 0, \'\', generateGoldPath=False); env.reset()\n                prefix = []; candidates = []\n                for action in gold[:args.max_prefix]:\n                    valid = sorted(set(env.get_valid_action_object_combinations()))\n                    if physical(action) and action in valid:\n                        candidates.append((list(prefix), action))\n                    obs, reward, done, info = env.step(action)\n                    prefix.append(action)\n                    if done: break\n                # Evenly sample trajectory positions rather than taking only its first actions.\n                if len(candidates) > args.per_task:\n                    indices = sorted({round(i*(len(candidates)-1)/(args.per_task-1)) for i in range(args.per_task)}) if args.per_task > 1 else [0]\n                    candidates = [candidates[i] for i in indices]\n                discovery[\'candidates\'] = [{\'prefix\': p, \'action\': a} for p,a in candidates]\n                save(folder/\'discovery.json\', discovery)\n                for prefix, action in candidates:\n                    cid = f\'P{len(results)+1:03d}\'; case = folder/cid; case.mkdir()\n                    print(cid, task, \':\', action, flush=True)\n                    before, t0 = replay(prefix); before2, t02 = replay(prefix)\n                    after, t1 = replay(prefix+[action]); after2, t12 = replay(prefix+[action])\n                    stable = before == before2 and after == after2 and t0 == t02 and t1 == t12\n                    record = {\'id\': cid, \'task\': task, \'variation\': 0, \'prefix\': prefix, \'candidate\': action,\n                        \'replay_stable\': stable, \'candidate_enumerated\': action in before[\'valid_actions\'],\n                        \'exported_world_changed\': projected(before) != projected(after),\n                        \'elapsed_moves\': after[\'moves\']-before[\'moves\'],\n                        \'valid_recovery_actions\': len(after[\'valid_actions\']), \'final_label\': None}\n                    with gzip.open(case/\'snapshots.json.gz\', \'wt\') as f:\n                        json.dump({\'before\': before, \'repeat_before\': before2, \'after\': after, \'repeat_after\': after2,\n                                   \'prefix_trace\': t0, \'candidate_trace\': t1}, f)\n                    if stable and not before[\'done\']:\n                        checked = search(replay, prefix, action, before, args.max_nodes, args.seconds_per_case)\n                        save(case/\'search.json\', checked)\n                        record[\'search\'] = {k:v for k,v in checked.items() if k != \'nodes\'}\n                    results.append(record); save(case/\'result.json\', record)\n                    save(out/\'progress.json\', {\'results\': results, \'errors\': errors})\n                    print(\'  replay stable:\', stable, \'|\', record.get(\'search\', {}).get(\'status\', \'search skipped\'), flush=True)\n            except Exception as exc:\n                errors.append({\'task\': task, \'error\': str(exc), \'type\': type(exc).__name__})\n                (folder/\'error.txt\').write_text(traceback.format_exc())\n                print(\'Task diagnostic error saved:\', task, str(exc), flush=True)\n            save(out/\'discoveries.json\', discoveries)\n    finally:\n        save(out/\'results.json\', results); save(out/\'summary.json\', summary())\n        if env is not None:\n            try: env.close()\n            except Exception: pass\n    print(\'SCIENCEWORLD PHYSICAL PILOT FINISHED — DIAGNOSTICS ONLY\')\n    print(json.dumps(summary(), indent=2))\n    return out\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--project\', default=\'/content/drive/MyDrive/ADNN_Irreversibility\')\n    p.add_argument(\'--per-task\', type=int, default=5)\n    p.add_argument(\'--max-prefix\', type=int, default=100)\n    p.add_argument(\'--max-nodes\', type=int, default=12)\n    p.add_argument(\'--seconds-per-case\', type=float, default=45)\n    args = p.parse_args()\n    assert min(args.per_task,args.max_prefix,args.max_nodes,args.seconds_per_case) > 0\n    run(args)\n')
print("Pilot script ready.")


In [ ]:
import runpy, sys
old_argv = sys.argv[:]
try:
    sys.argv = ['/content/37_scienceworld_physical_pilot_colab.py']
    runpy.run_path(sys.argv[0], run_name='__main__')
finally:
    sys.argv = old_argv


## What to send back
Copy the final summary into the project chat. If any task reports an error, include that message. Do not start large dataset generation yet. The output reports diagnostic checks, not fulfilment of the ScienceWorld requirement.

The 45-second search budget is checked between simulator calls, not a hard interruption of Java calls. Task discovery and replay add time. Stop the cell if it becomes unresponsive; already saved results remain on Drive.

API reference: https://github.com/allenai/ScienceWorld/blob/e8216d6044e8e39be9fcb185e3b2dfb602584b52/scienceworld/scienceworld.py
